# 13 PBS：在优先级偏序上搜索


本课按百科条目写。PBS 不预先固定全序，在「谁优先于谁」的约束上搜索。调度台在最后。

---

## 1. 定义

**Priority-Based Search**（Ma et al., AAAI 2019）：高层在 **成对优先级约束** 上 DFS/搜索；底层仍是 HCA\*。

撞车 (a,b) 时分支：

- 子节点：a 优先于 b（a≺b 表示 a 先规划）
- 子节点：b 优先于 a

约束组成 DAG，拓扑排序得到全序再跑 HCA。出现环（互相要求对方先走）则剪枝。

- 输入：图、起终点。
- 输出：某个可行全序下的 HCA 路径，以及用到的成对约束。

---

## 2. 保证

比固定 FIFO 更能找到可行序。底层仍是优先规划 ⇒ **不完全**（有向死锁、空位不够）。**不保证 SOC 最优**（本实现记下搜到的可行解中较好者，不是 CBS 那种可采纳高层）。

对比：CBS 在时空约束上分支，最优；PBS 在优先级上分支，更快但不最优。

---

## 3. 不变量

- 约束集只增不减。
- 拓扑排序失败 ⇔ 约束有环 ⇔ 该支剪掉。
- `seen` 避免同一偏序重复搜。

k 车全序 k! 种，PBS 通常只展开「真发生冲突」的那几对，远小于 k!。

---

## 4. 伪代码

```
search(constraints):
    order ← Kahn拓扑(constraints)
    若有环: return
    r ← HCA(order)
    若 r 可行: 记录若更好则更新 best; return
    (a,b) ← r 中第一对冲突
    search(constraints ∪ {a≺b})
    search(constraints ∪ {b≺a})
```

两车时退化为：先试一种序，失败再试反过来——下面就先写这个。

---

## 5. 手算

两车开网格平行走：HCA 第一种序就成功，PBS 不分支。

两车必须「谁让谁」才过得去：第一种序失败，冲突对 (a,b)，分支后某序成功。

有向 L→R 对头：两种序都失败，PBS 也失败。


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


## 6. 实现：两车先穷举序，再看一般约束搜索


In [ ]:
from evals.scenarios import two_agent_tunnel
from lib.algos.prioritized import hca
from lib.maps import directed_deadlock_map


def pbs_two(graph, starts, goals):
    agents = list(starts)
    a, b = agents[0], agents[1]
    best = None
    for order in ([a, b], [b, a]):
        r = hca(graph, starts, goals, order=order)
        print("试序", order, "found", r["found"], "SOC", r.get("soc"))
        if r["found"] and (best is None or r["soc"] < best["soc"]):
            best = r
            best["constraints"] = [(order[0], order[1])]
    if best is None:
        return {"found": False, "paths": {}, "soc": float("inf")}
    return best


g, starts, goals = two_agent_tunnel()
r2 = pbs_two(g, starts, goals)
print("两车 PBS", r2["found"], r2["soc"], r2.get("constraints"))
print(
    "有向死锁",
    pbs_two(directed_deadlock_map(), {"a": "L", "b": "R"}, {"a": "R", "b": "L"})["found"],
)


两车版已经能说明「失败就换序」。k>2 时不能枚举 k!，要用约束 DFS + Kahn。


In [ ]:
from collections import defaultdict, deque
from lib.agents import vertex_conflicts, edge_conflicts


def kahn(agents, constraints):
    # constraints: 列表 (hi, lo)，hi 先走
    succ = defaultdict(list)
    indeg = {a: 0 for a in agents}
    for hi, lo in constraints:
        succ[hi].append(lo)
        indeg[lo] += 1
    q = deque([a for a in agents if indeg[a] == 0])
    order = []
    while q:
        n = q.popleft()
        order.append(n)
        for s in succ[n]:
            indeg[s] -= 1
            if indeg[s] == 0:
                q.append(s)
    if len(order) != len(agents):
        return None
    return order


print("a≺b, b≺c =>", kahn(["a", "b", "c"], [("a", "b"), ("b", "c")]))
print("环 a≺b, b≺a =>", kahn(["a", "b"], [("a", "b"), ("b", "a")]))


In [ ]:
from lib.algos.prioritized import pbs

r = pbs(g, starts, goals)
print("库 PBS", r["found"], r["soc"], r.get("constraints"))


## 7. 逐行（库 `pbs.search`）

| 块 | 作用 |
|----|------|
| indeg / 队列 | Kahn 拓扑 |
| 环 | 互相要求对方先走 |
| vertex_conflicts | 从失败的 HCA 路径找要分支的一对 |
| seen + depth | 防同一偏序、防无限递归 |

## 常见 bug

1. 约束写成无向，无法排序。
2. 不记 seen，同一偏序重复搜。
3. 冲突双方已经有优先级仍再分支，造环。
4. 把 PBS 当最优算法去比 CBS 的 SOC。

## 条目小结

| 项目 | 内容 |
|------|------|
| 高层 | 成对优先级 DAG |
| 底层 | HCA\* |
| 最优 | 否 |
| 完备 | 否（同 HCA） |
| 下一课 | 窗口滚起来：RHCR |

## 练习

1. 三车约束 a≺b、c≺b，写出所有合法全序。
2. 为何「HCA 失败才分支」比「预先枚举全序」省？
3. 有向死锁上 PBS 失败，换 CBS 会成功吗？先想再跑 15 课。


In [ ]:
from lib.studio.widget import PlannerStudio
studio = PlannerStudio()
studio.show_multi(g.to_studio(), r, "PBS")
studio
